# Notebook 05 — Complete the original fixed protocol

Select **T4 GPU** and run the one code cell. Keep the existing `SensorRepair_ICDSIS` Drive project and Notebook 01–04 outputs. No new download or manual data movement is required.

**Scope:** DSADS four participant folds × seeds 2026, 2027, 2028 (12 configurations); HAR official participant test × the same three seeds (3 configurations). The two existing fold-0/seed-2026 test results are imported with exact summary and participant parity checks. The remaining **13 configurations** are trained and evaluated. No completed pilot is discarded or retrained.

**Method stays fixed.** Each configuration independently fits its neural models on its train partition, its benefit/harm heads on its policy partition, and its thresholds on its validation partition using the existing grids and empirical constraints. All thresholds are committed before that configuration's test inference. Stuck faults are absent from training and threshold selection. The primary comparator families remain residual masking for DSADS and keep for HAR; every baseline and ablation is also reported.

**Resume:** a 90-minute session budget pauses at an epoch or condition boundary. Rerun this same cell to continue. Each completed training epoch, action condition, model head and configuration is saved. A Colab disconnection can replay the last unfinished epoch or condition. Keep the same software versions and GPU type when resuming. No number of new training runs or GPU time is guaranteed on the free tier.

**Reporting:** per-run metrics and participant results are saved. A complete DSADS seed is the equal-weight mean of its four fold scores; a complete HAR seed uses its official test score. The final summary averages three seeds and shows their sample SD. Fault realizations also vary with run seed, following the original RNG rule. SD is descriptive, not a confidence interval. A partially completed seed never enters the cross-seed aggregate. Original windows, fault views, folds and seeds are not treated as independent people.

The initial test results are already known: DSADS improves on known faults but exceeds the 1% empirical harm target, has participant heterogeneity, and trails confidence selection on stuck faults; HAR's original selected policy is constant keep. These outcomes are retained. This extension completes the earlier protocol without changing the method to respond to those outcomes. It makes no guarantee of a performance gain, a deployment harm bound, significance, or paper acceptance.

If the notebook pauses, share its output to check progress, or rerun it until it prints **NOTEBOOK 05 COMPLETE** and **15/15**. Then share the final output and the three named result files. Earlier notebooks and their outputs remain untouched.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import gc
import hashlib
import json
import math
import random
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

PROJECT = Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT = PROJECT / 'outputs' / 'notebook05_v1'
LOCAL = Path('/content/sensorrepair_notebook05_v1')
REQUIRE_GPU = True
SESSION_MINUTES = 90  # Rerun this same cell after a pause; finished runs and epochs are reused.
CODE_VERSION = 'sensorrepair_nb05_v1'
CELL_SIGNATURE = '14142e7667994ec0266447a0fc8b3378a9921d2a2552384c8961a332f50e3492'
CLASSIFIER_EPOCHS = 30
RECONSTRUCTOR_EPOCHS = 25
PATIENCE = 6
BATCH_SIZE = 128
INFERENCE_BATCH_SIZE = 128
LEARNING_RATE = 0.001
MAX_ITER = 160
CLEAN_F1_TOLERANCE = 0.005
MAX_CLEAN_HARM = 0.01
MAX_CORRUPTED_HARM = 0.01
UTILITY_THRESHOLDS = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 2.0]
HARM_PENALTIES = [1.0, 2.0, 4.0]
CONFIDENCE_MARGINS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 2.0]
RESIDUAL_THRESHOLDS = [0.0, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 128.0, None]
FAMILIES = ['harm_aware_policy', 'benefit_only_policy', 'confidence_margin_tuned',
            'residual_mask_tuned', 'residual_reconstruct_tuned']
BASELINES = ['keep', 'mask_suspect', 'reconstruct_suspect', 'residual_gate_mask',
             'residual_gate_reconstruct', 'confidence_select', 'oracle_diagnostic']
METHODS = BASELINES + FAMILIES
CONDITIONS = [('clean', None, None)] + [(f'{fault}_{i+1}', fault, i)
             for fault in ['dropout','bias','gain','stuck'] for i in range(3)]
SCOPES = {'clean':['clean'],
          'known_faults':[name for name,fault,_ in CONDITIONS if fault in ['dropout','bias','gain']],
          'unseen_stuck':[name for name,fault,_ in CONDITIONS if fault=='stuck']}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, then rerun this same cell.')
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
SESSION_DEADLINE = time.monotonic() + SESSION_MINUTES*60
print('NOTEBOOK 05 | original full protocol | one executable cell',flush=True)
print('Device:',torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU','| PyTorch:',torch.__version__)
print('DSADS: 4 folds x 3 seeds | HAR: official test x 3 seeds | 15 total configurations.',flush=True)
print('Completed fold_0 / seed 2026 results will be imported. Training and selection rules remain fixed.',flush=True)



def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def save_checkpoint(path, state):
    path = Path(path)
    temporary = LOCAL / (path.name + '.tmp')
    torch.save(state, temporary)
    partial = path.with_name(path.name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


class ActivityCNN(nn.Module):
    def __init__(self, channels, classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 7, padding=3), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.GroupNorm(8, 96), nn.GELU(),
            nn.Conv1d(96, 128, 5, stride=2, padding=2), nn.GroupNorm(8, 128), nn.GELU(),
            nn.AvgPool1d(kernel_size=8, stride=8), nn.Flatten(),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, classes))

    def forward(self, values, available=None):
        if available is None:
            available = torch.ones_like(values)
        return self.encoder(torch.cat([values, available], dim=1))


class SensorReconstructor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(channels * 2, 64, 5, padding=2), nn.GELU(),
            nn.Conv1d(64, 96, 5, padding=2), nn.GELU(),
            nn.Conv1d(96, 64, 5, padding=2), nn.GELU(), nn.Conv1d(64, channels, 1))

    def forward(self, values, available):
        return self.network(torch.cat([values, available], dim=1))


def mask_groups(values, groups, rng):
    masked = np.array(values, dtype=np.float32, copy=True)
    available = np.ones_like(masked)
    for i in range(len(masked)):
        channels = groups[int(rng.integers(len(groups)))]['channels']
        masked[i, channels, :] = 0.0
        available[i, channels, :] = 0.0
    return masked, available


def classifier_epoch_inputs(clean, groups, seed):
    rng = np.random.default_rng(seed)
    values = np.array(clean, copy=True)
    available = np.ones_like(values)
    choices = rng.random(len(values))
    faults = np.flatnonzero((choices >= 0.35) & (choices < 0.80))
    masks = np.flatnonzero(choices >= 0.80)
    values[faults] = corrupt_signals(clean[faults], groups, rng)
    masked, indicators = mask_groups(clean[masks], groups, rng)
    values[masks], available[masks] = masked, indicators
    return values, available


@torch.inference_mode()
def classifier_probabilities(model, values, available=None):
    model.eval()
    outputs = []
    for start in range(0, len(values), INFERENCE_BATCH_SIZE):
        xb = torch.as_tensor(values[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        mb = None if available is None else torch.as_tensor(available[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        outputs.append(model(xb, mb).softmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)


@torch.inference_mode()
def reconstruction_validation(model, values, groups):
    model.eval()
    total, count = 0.0, 0
    for start in range(0, len(values), INFERENCE_BATCH_SIZE):
        clean = torch.as_tensor(values[start:start + INFERENCE_BATCH_SIZE], device=DEVICE)
        for group in groups:
            channels = group['channels']
            observed, available = clean.clone(), torch.ones_like(clean)
            observed[:, channels, :], available[:, channels, :] = 0.0, 0.0
            prediction = model(observed, available)
            errors = (prediction[:, channels, :] - clean[:, channels, :]).square()
            total += float(errors.sum().item())
            count += errors.numel()
    return total / count


def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def fit_model(kind, model, clean_train, train_y, validation, val_y, groups, classes, run, signature, stage_seed):
    seed_everything(stage_seed)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
    last_file, best_file = run / f'{kind}_last.pt', run / f'{kind}_best.pt'
    maximum = CLASSIFIER_EPOCHS if kind == 'classifier' else RECONSTRUCTOR_EPOCHS
    history, first_epoch, stale, best_score = [], 1, 0, -1e30
    if last_file.exists():
        checkpoint = load_checkpoint(last_file)
        assert checkpoint['signature'] == signature, 'Checkpoint configuration changed; use a new output version.'
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        history, first_epoch = checkpoint['history'], checkpoint['epoch'] + 1
        stale, best_score = checkpoint['stale'], checkpoint['best_score']
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda' and checkpoint['cuda_rng']:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        if checkpoint['complete']:
            best = load_checkpoint(best_file)
            assert best['signature'] == signature
            model.load_state_dict(best['model'])
            print(f'  {kind}: reused completed checkpoint (best epoch {best["epoch"]}).', flush=True)
            return model, history
        print(f'  {kind}: resuming at epoch {first_epoch}.', flush=True)
    counts = np.bincount(train_y, minlength=classes)
    weights = 1.0 / np.sqrt(np.maximum(counts, 1))
    weights = torch.as_tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)
    validation_cases = [validation]
    if kind == 'classifier':
        for number, fault in enumerate(['dropout', 'bias', 'gain']):
            validation_cases.append(corrupt_signals(validation, groups, np.random.default_rng(stage_seed + 800 + number), fault, 1))
    reconstruction_indices = np.random.default_rng(stage_seed + 500).permutation(len(validation))[:256]
    elapsed_before = history[-1]['cumulative_seconds'] if history else 0.0
    started = time.perf_counter()
    for epoch in range(first_epoch, maximum + 1):
        epoch_started = time.perf_counter()
        lr = LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * (epoch - 1) / maximum)))
        for group in optimizer.param_groups:
            group['lr'] = lr
        if kind == 'classifier':
            values, available = classifier_epoch_inputs(clean_train, groups, stage_seed + epoch * 1000)
            target = torch.from_numpy(train_y)
        else:
            values, available = mask_groups(clean_train, groups, np.random.default_rng(stage_seed + epoch * 1000))
            target = torch.from_numpy(clean_train)
        generator = torch.Generator().manual_seed(stage_seed + epoch)
        loader = DataLoader(TensorDataset(torch.from_numpy(values), torch.from_numpy(available), target),
                            batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                            num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
        model.train()
        loss_sum, seen = 0.0, 0
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(xb, mb)
            if kind == 'classifier':
                log_probability = prediction.log_softmax(dim=1).gather(1, yb[:, None]).squeeze(1)
                example_weight = weights[yb]
                loss = -(log_probability * example_weight).sum() / example_weight.sum()
            else:
                missing = 1.0 - mb
                loss = ((prediction - yb).square() * missing).sum() / missing.sum().clamp_min(1.0)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite {kind} loss; the last completed epoch is saved.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += float(loss.item()) * len(xb)
            seen += len(xb)
        if kind == 'classifier':
            scores = [macro_f1(val_y, classifier_probabilities(model, case).argmax(axis=1), classes)
                      for case in validation_cases]
            score, val_measure = float(np.mean(scores)), float(np.mean(scores))
        else:
            val_measure = reconstruction_validation(model, validation[reconstruction_indices], groups)
            score = -val_measure
        improved = score > best_score + 1e-6
        if improved:
            best_score, stale = score, 0
            save_checkpoint(best_file, {'signature': signature, 'epoch': epoch, 'model': cpu_state(model),
                                        'validation_measure': val_measure})
        else:
            stale += 1
        cuda_sync()
        history.append({'epoch': epoch, 'train_loss': loss_sum / seen, 'validation_measure': val_measure,
                        'epoch_seconds': time.perf_counter() - epoch_started,
                        'cumulative_seconds': elapsed_before + time.perf_counter() - started, 'learning_rate': lr})
        complete = epoch == maximum or stale >= PATIENCE
        save_checkpoint(last_file, {'signature': signature, 'epoch': epoch, 'model': cpu_state(model),
                                    'optimizer': optimizer.state_dict(), 'best_score': best_score,
                                    'stale': stale, 'history': history, 'complete': complete,
                                    'torch_rng': torch.get_rng_state(),
                                    'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else []})
        save_csv(run / f'{kind}_history.csv', pd.DataFrame(history))
        if epoch == 1 or epoch % 5 == 0 or complete:
            label = 'validation mean Macro-F1' if kind == 'classifier' else 'validation masked MSE'
            print(f'  {kind} epoch {epoch:02d}/{maximum} | loss {loss_sum / seen:.4f} | {label} {val_measure:.4f}', flush=True)
        del values, available, target, loader
        if not complete:
            check_budget()
        if complete:
            break
    best = load_checkpoint(best_file)
    assert best['signature'] == signature
    model.load_state_dict(best['model'])
    return model, history


@torch.inference_mode()
def action_outputs(classifier, reconstructor, values, groups, classes):
    classifier.eval()
    reconstructor.eval()
    number = len(values)
    group_count = len(groups)
    probabilities = np.empty((number, 1 + 2 * group_count, classes), dtype=np.float32)
    residuals = np.empty((number, group_count), dtype=np.float32)
    group_statistics = np.empty((number, group_count, 4), dtype=np.float32)
    cuda_sync()
    started = time.perf_counter()
    for start in range(0, number, INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, number)
        observed = torch.as_tensor(values[start:stop], device=DEVICE)
        probabilities[start:stop, 0] = classifier(observed).softmax(dim=1).cpu().numpy()
        for index, group in enumerate(groups):
            channels = group['channels']
            masked, available = observed.clone(), torch.ones_like(observed)
            masked[:, channels, :], available[:, channels, :] = 0.0, 0.0
            reconstruction = reconstructor(masked, available)
            repaired = observed.clone()
            repaired[:, channels, :] = reconstruction[:, channels, :]
            probabilities[start:stop, 1 + index] = classifier(masked, available).softmax(dim=1).cpu().numpy()
            probabilities[start:stop, 1 + group_count + index] = classifier(repaired).softmax(dim=1).cpu().numpy()
            residuals[start:stop, index] = (observed[:, channels, :] - reconstruction[:, channels, :]).square().mean(dim=(1, 2)).cpu().numpy()
            signal = values[start:stop, channels, :]
            group_statistics[start:stop, index] = np.stack([
                np.abs(signal).mean(axis=(1, 2)), signal.std(axis=(1, 2)),
                np.abs(np.diff(signal, axis=2)).mean(axis=(1, 2)),
                (signal == 0).mean(axis=(1, 2))], axis=1)
    cuda_sync()
    seconds = time.perf_counter() - started
    assert np.isfinite(probabilities).all() and np.isfinite(residuals).all()
    assert np.allclose(probabilities.sum(axis=2), 1.0, atol=1e-5)
    return probabilities, residuals, group_statistics, seconds


def corrupt_signals(values, groups, rng, fault=None, severity=None):
    # Fault locations and severity metadata are never returned as model features.
    result = np.array(values, dtype=np.float32, copy=True)
    for i in range(len(result)):
        kind = fault if fault is not None else str(rng.choice(['dropout', 'bias', 'gain']))
        level = int(severity) if severity is not None else int(rng.integers(0, 3))
        group = groups[int(rng.integers(len(groups)))]['channels']
        if kind == 'dropout':
            fraction = protocol['fault_severities']['dropout']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            result[i, group, start:start + length] = 0.0
        elif kind == 'bias':
            amount = protocol['fault_severities']['bias']['absolute_offset_in_training_std'][level]
            signs = rng.choice([-1.0, 1.0], size=len(group)).astype(np.float32)
            result[i, group, :] += np.float32(amount) * signs[:, None]
        elif kind == 'gain':
            factor = float(protocol['fault_severities']['gain']['factor'][level])
            factor = factor if rng.random() < 0.5 else 1.0 / factor
            result[i, group, :] *= np.float32(factor)
        elif kind == 'stuck':
            fraction = protocol['fault_severities']['stuck']['time_fraction'][level]
            length = max(1, int(round(result.shape[2] * fraction)))
            start = int(rng.integers(0, result.shape[2] - length + 1))
            first = result[i, group, start].copy()
            result[i, group, start:start + length] = first[:, None]
        else:
            raise ValueError(f'Unknown fault: {kind}')
    return result


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def outcome_targets(pack):
    predictions = pack['probabilities'].argmax(axis=-1)
    correct = predictions == pack['y'][None, :, None]
    original = correct[:, :, 0:1]
    fix = (~original & correct[:, :, 1:]).reshape(-1).astype(np.int8)
    harm = (original & ~correct[:, :, 1:]).reshape(-1).astype(np.int8)
    assert not np.any(fix & harm)
    return fix, harm


def fit_head(name, X, targets, run, signature, feature_names):
    path = run / f'{name}_head.joblib'
    if path.exists():
        payload = joblib.load(path)
        assert payload['signature'] == signature and payload['feature_names'] == feature_names, 'Stale policy model.'
        print(f'  {name} head: reused saved model.', flush=True)
        return payload['model'], payload['training_seconds']
    started = time.perf_counter()
    if len(np.unique(targets)) == 1:
        model = {'constant_probability': float(targets[0])}
    else:
        model = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=MAX_ITER,
                max_leaf_nodes=15, max_depth=4, min_samples_leaf=40, l2_regularization=5.0,
                max_bins=64, categorical_features=None, early_stopping=False,
                random_state=SEED + (1 if name == 'harm' else 0))
        model.fit(X, targets)
    seconds = time.perf_counter() - started
    atomic_joblib(path, {'signature': signature, 'model': model, 'feature_names': feature_names,
                        'training_seconds': seconds, 'positive_targets': int(targets.sum()), 'training_rows': len(targets)})
    print(f'  {name} head fitted | positive outcomes {int(targets.sum())}/{len(targets)} | {seconds:.1f}s', flush=True)
    return model, seconds


def predict_head(model, X):
    if isinstance(model, dict):
        return np.full(len(X), model['constant_probability'], dtype=np.float32)
    indices = np.flatnonzero(model.classes_ == 1)
    return model.predict_proba(X)[:, int(indices[0])].astype(np.float32) if len(indices) else np.zeros(len(X), dtype=np.float32)


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer


def baseline_choices(pack, scale):
    p, residual = pack['probabilities'], pack['residuals']
    D, N, A, K = p.shape
    G = residual.shape[-1]
    ratio = residual / scale[None, None, :]
    suspect = ratio.argmax(axis=-1)
    gate = ratio.max(axis=-1) > 1.0
    original_correct = p.argmax(axis=-1) == pack['y'][None, :, None]
    return {
        'keep': np.zeros((D, N), dtype=np.int64),
        'mask_suspect': 1 + suspect,
        'reconstruct_suspect': 1 + G + suspect,
        'residual_gate_mask': np.where(gate, 1 + suspect, 0),
        'residual_gate_reconstruct': np.where(gate, 1 + G + suspect, 0),
        'confidence_select': p.max(axis=-1).argmax(axis=-1),
        'oracle_diagnostic': original_correct.argmax(axis=-1)
    }


def choose_actions(pack, probability_fix, probability_harm, scale, settings):
    family = settings['family']
    D, N, A, K = pack['probabilities'].shape
    if family in ['harm_aware_policy', 'benefit_only_policy']:
        scores = probability_fix - settings['harm_penalty'] * probability_harm
        selected = scores.argmax(axis=-1)
        return np.where(scores.max(axis=-1) > settings['threshold'], selected + 1, 0)
    if family == 'confidence_margin_tuned':
        confidence = pack['probabilities'].max(axis=-1)
        selected = confidence.argmax(axis=-1)
        return np.where(confidence.max(axis=-1) > confidence[:, :, 0] + settings['threshold'], selected, 0)
    if settings['threshold'] is None:
        return np.zeros((D, N), dtype=np.int64)
    ratio = pack['residuals'] / scale[None, None, :]
    selected = ratio.argmax(axis=-1)
    offset = 1 if family == 'residual_mask_tuned' else 1 + ratio.shape[-1]
    return np.where(ratio.max(axis=-1) > settings['threshold'], selected + offset, 0)


def candidates_for(family):
    if family == 'harm_aware_policy':
        return [{'family': family, 'harm_penalty': penalty, 'threshold': threshold}
                for penalty in HARM_PENALTIES for threshold in UTILITY_THRESHOLDS]
    if family == 'benefit_only_policy':
        return [{'family': family, 'harm_penalty': 0.0, 'threshold': threshold} for threshold in UTILITY_THRESHOLDS]
    thresholds = CONFIDENCE_MARGINS if family == 'confidence_margin_tuned' else RESIDUAL_THRESHOLDS
    return [{'family': family, 'threshold': threshold} for threshold in thresholds]


def select_settings(pack, probability_fix, probability_harm, scale, family, clean_keep_f1):
    best, trials = None, []
    for number, settings in enumerate(candidates_for(family)):
        actions = choose_actions(pack, probability_fix, probability_harm, scale, settings)
        summary = aggregate_rows(evaluate_choices(pack, actions, family))
        feasible = (summary['clean_macro_f1'] >= clean_keep_f1 - CLEAN_F1_TOLERANCE - 1e-12
                    and summary['clean_harmful_rate'] <= MAX_CLEAN_HARM + 1e-12
                    and summary['harmful_rate'] <= MAX_CORRUPTED_HARM + 1e-12)
        record = {'trial': number, 'family': family, 'settings_json': json.dumps(settings),
                  'feasible': bool(feasible), **summary}
        trials.append(record)
        rank = (summary['macro_f1'], summary['clean_macro_f1'], -summary['harmful_rate'], -summary['intervention_rate'])
        if feasible and (best is None or rank > best['rank']):
            best = {'rank': rank, 'settings': settings, 'actions': actions, 'summary': summary}
    assert best is not None, 'The explicit keep-only candidate must be feasible.'
    return best, trials




class PauseStudy(Exception):
    pass


def check_budget():
    if time.monotonic() >= SESSION_DEADLINE:
        raise PauseStudy('Session time budget reached; all completed epochs and conditions are saved.')


STAGES = {'prepared':PROJECT/'processed'/'v1',
          'nb02':PROJECT/'outputs'/'notebook02_v1',
          'nb03':PROJECT/'outputs'/'notebook03_v1',
          'nb04':PROJECT/'outputs'/'notebook04_v1'}
INVENTORY_NAMES = {'prepared':'artifact_inventory.json','nb02':'notebook02_artifact_inventory.json',
                   'nb03':'notebook03_artifact_inventory.json','nb04':'notebook04_artifact_inventory.json'}
inventories,inventory_hashes={},{}
for stage,root in STAGES.items():
    path=root/INVENTORY_NAMES[stage]
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Keep all Notebook 01–04 outputs in the existing project.')
    inventory_hashes[stage]=file_sha256(path)
    records=json.loads(path.read_text())
    inventories[stage]={r['filename']:r for r in records}
    assert len(records)==len(inventories[stage])


def source_file(stage,relative):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    expected=inventories[stage][relative]['sha256']
    target=LOCAL/'inputs'/stage/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:
        return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(STAGES[stage]/relative,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {stage}/{relative}'
    os.replace(partial,target)
    return target


def source_json(stage,relative):
    return json.loads(source_file(stage,relative).read_text())


protocol=source_json('prepared','protocol.json')
assert protocol['model_seeds']==[2026,2027,2028]
assert protocol['development_fault_types']==['dropout','bias','gain'] and protocol['held_out_fault_type']=='stuck'
assert protocol['normalization_fit_split']=='train'
completion=json.loads((STAGES['prepared']/'notebook01_complete.json').read_text())
assert completion['ready_for_training']
assert completion['artifact_inventory_sha256']==inventory_hashes['prepared']
assert completion['protocol_sha256']==inventories['prepared']['protocol.json']['sha256']
prior_report=source_json('nb04','notebook04_run_report.json')
assert prior_report['datasets_completed']==['dsads','har'] and prior_report['validation_parity_passed']
assert prior_report['fold']=='fold_0' and prior_report['seed']==2026
prior_by_dataset={r['dataset']:r for r in prior_report['results']}
PRIMARY={name:prior_by_dataset[name]['frozen_primary_comparator'] for name in ['dsads','har']}
assert PRIMARY=={'dsads':'residual_gate_mask','har':'keep'}
for dataset in ['dsads','har']:
    run=f'{dataset}_fold_0_seed_2026'
    old2=source_json('nb02',run+'/run_configuration.json')['settings']
    old3=source_json('nb03',run+'/run_configuration.json')['configuration']
    for key,value in {'classifier_epochs':CLASSIFIER_EPOCHS,'reconstructor_epochs':RECONSTRUCTOR_EPOCHS,
                      'patience':PATIENCE,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE}.items():
        assert old2[key]==value,f'Original training setting changed: {key}'
    assert old2['protocol_sha256']==completion['protocol_sha256']
    for filename,digest in old2['input_sha256'].items():
        assert inventories['prepared'][filename]['sha256']==digest
    assert old3['max_iter']==MAX_ITER
    assert old3['harm_penalties']==HARM_PENALTIES and old3['utility_thresholds']==UTILITY_THRESHOLDS
    assert old3['confidence_margins']==CONFIDENCE_MARGINS and old3['residual_thresholds']==RESIDUAL_THRESHOLDS
    assert old3['empirical_selection_guards']=={'clean_f1_tolerance':CLEAN_F1_TOLERANCE,
        'max_clean_harm':MAX_CLEAN_HARM,'max_corrupted_harm':MAX_CORRUPTED_HARM}
    assert old3['model']=={'learning_rate':.05,'max_leaf_nodes':15,'max_depth':4,'min_samples_leaf':40,
                           'l2_regularization':5.,'max_bins':64,'early_stopping':False}
    assert old2['torch_version']==str(torch.__version__) and old3['sklearn_version']==sklearn.__version__, (
        'Use the original Notebook 02/03 PyTorch and scikit-learn versions for this study extension. '
        f'Expected torch={old2["torch_version"]}, sklearn={old3["sklearn_version"]}.')

SEEDS=protocol['model_seeds']
# Finish every seed's participant coverage before moving to the next seed.
JOBS=[{'dataset':dataset,'fold':fold,'seed':seed}
      for seed in SEEDS for dataset in ['dsads','har']
      for fold in ([f'fold_{i}' for i in range(4)] if dataset=='dsads' else ['fold_0'])]
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'jobs':JOBS,
      'source_inventory_hashes':inventory_hashes,'protocol_sha256':completion['protocol_sha256'],
      'training':{'classifier_epochs':CLASSIFIER_EPOCHS,'reconstructor_epochs':RECONSTRUCTOR_EPOCHS,
                  'patience':PATIENCE,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,
                  'architectures':'exact Notebook 02 CNN and group reconstructor; unchanged losses and augmentation'},
      'policy':{'max_iter':MAX_ITER,'learning_rate':.05,'max_leaf_nodes':15,'max_depth':4,
                'min_samples_leaf':40,'l2_regularization':5.,'max_bins':64,'early_stopping':False,
                'harm_penalties':HARM_PENALTIES,'utility_thresholds':UTILITY_THRESHOLDS,
                'confidence_margins':CONFIDENCE_MARGINS,'residual_thresholds':RESIDUAL_THRESHOLDS,
                'clean_f1_tolerance':CLEAN_F1_TOLERANCE,'max_clean_harm':MAX_CLEAN_HARM,
                'max_corrupted_harm':MAX_CORRUPTED_HARM},
      'selection':'independent per run: train neural models on train; fit heads on policy; select thresholds on validation only',
      'test_use':'evaluate only after saving selected policy; never select hyperparameters on test',
      'known_pilot_test_seen':True,'pilot_result_retained':'import both fold_0 seed 2026 results without retraining',
      'primary_comparators':PRIMARY,'all_baselines_and_ablations_retained':METHODS,
      'conditions':CONDITIONS,'fault_specification':protocol['fault_severities'],
      'fault_rng':'run model seed + crc32(dataset|split|condition), identical to Notebook 02/04 rule',
      'seed_variation_includes_fault_realization':True,
      'aggregation':'per-condition test Macro-F1; mean nine known or three stuck conditions; equal mean four DSADS folds within seed; mean three seeds',
      'population_independent_units':{'dsads':8,'har':9},
      'seed_sd_is_not_confidence_interval':True,'no_statistical_significance_claim':True,
      'empirical_harm_target_is_not_certified_bound':True,
      'environment':{'torch':str(torch.__version__),'sklearn':sklearn.__version__,'numpy':np.__version__,
                     'device':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'}}
STUDY_SIGNATURE=json_signature(plan)
manifest=OUTPUT/'notebook05_study_manifest.json'
if manifest.exists():
    assert json.loads(manifest.read_text())['signature']==STUDY_SIGNATURE, (
        'The fixed study inputs, method or environment changed. Restore the original configuration; do not overwrite completed test results.')
else:
    save_json(manifest,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Full study plan frozen:',STUDY_SIGNATURE[:16],flush=True)


def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'


def job_signature(job):
    return json_signature({'study':STUDY_SIGNATURE,**job})


def load_dataset(job,need_raw=True):
    dataset,fold=job['dataset'],job['fold']
    with np.load(source_file('prepared',f'{dataset}_signals.npz'),allow_pickle=False) as saved:
        data={key:saved[key] for key in ['y','subjects','window_ids']}
        if need_raw:
            data['raw']=saved['X']
    with np.load(source_file('prepared',f'{dataset}_splits.npz'),allow_pickle=False) as saved:
        data['splits']={split:saved[f'{fold}_{split}'] for split in ['train','policy','validation','test']}
    with np.load(source_file('prepared',f'{dataset}_{fold}_normalization.npz'),allow_pickle=False) as saved:
        data['mean'],data['std']=saved['mean'],saved['std']
    for split,indices in data['splits'].items():
        assert len(indices)==len(np.unique(indices))
        for other,second in data['splits'].items():
            if split!=other:
                assert not set(data['subjects'][indices].tolist())&set(data['subjects'][second].tolist())
    all_indices=np.concatenate(list(data['splits'].values()))
    assert np.array_equal(np.sort(all_indices),np.arange(len(data['y'])))
    data['groups']=protocol['datasets'][dataset]['sensor_groups']
    data['classes']=len(protocol['datasets'][dataset]['class_names'])
    assert np.array_equal(np.unique(data['y']),np.arange(data['classes']))
    assert np.isfinite(data['mean']).all() and np.all(data['std']>0)
    return data


def normalized(data,split):
    return np.ascontiguousarray((data['raw'][data['splits'][split]]-data['mean'][None,:,None])/
                               data['std'][None,:,None],dtype=np.float32)


def case_caches(job,split,values,data,classifier,reconstructor,run,signature):
    indices=data['splits'][split]
    groups=data['groups']
    names=np.asarray(['keep']+[f'mask:{g["name"]}' for g in groups]+[f'reconstruct:{g["name"]}' for g in groups])
    paths=[]
    conditions=CONDITIONS if split=='test' else CONDITIONS[:10]
    for condition,fault,severity in conditions:
        check_budget()
        path=run/'action_cache'/f'{split}__{condition}.npz'
        digest_path=path.with_name(path.name+'.sha256.json')
        if path.is_file() and digest_path.is_file():
            assert json.loads(digest_path.read_text())=={'signature':signature,'sha256':file_sha256(path)}, 'Cache hash changed.'
            with np.load(path,allow_pickle=False) as saved:
                assert saved['signature'].item()==signature
                assert saved['condition'].item()==condition and saved['split'].item()==split
                for key,expected in [('y',data['y'][indices]),('subjects',data['subjects'][indices]),
                    ('window_ids',data['window_ids'][indices]),('source_indices',indices),('action_names',names)]:
                    assert np.array_equal(saved[key],expected),f'Cache alignment failed: {key}'
        else:
            token=zlib.crc32(f'{job["dataset"]}|{split}|{condition}'.encode())
            observed=values if fault is None else corrupt_signals(values,groups,np.random.default_rng(job['seed']+token),fault,severity)
            p,r,s,seconds=action_outputs(classifier,reconstructor,observed,groups,data['classes'])
            save_npz(path,signature=np.asarray(signature),probabilities=p,reconstruction_residuals=r,
                     observed_group_statistics=s,y=data['y'][indices],subjects=data['subjects'][indices],
                     window_ids=data['window_ids'][indices],source_indices=indices,action_names=names,
                     condition=np.asarray(condition),split=np.asarray(split),full_candidate_inference_seconds=np.asarray(seconds))
            save_json(digest_path,{'signature':signature,'sha256':file_sha256(path)})
            del observed,p,r,s
        paths.append(path)
    print(f'  {split}: {len(paths)} action conditions ready.',flush=True)
    return pack_from_caches(paths,signature)


def head_outputs(pack,scale,feature_names,benefit,harm):
    X,names=policy_features(pack['probabilities'],pack['residuals'],pack['statistics'],scale)
    assert names==feature_names
    shape=pack['probabilities'].shape[:2]+(pack['probabilities'].shape[2]-1,)
    fix=predict_head(benefit,X).reshape(shape)
    risk=predict_head(harm,X).reshape(shape)
    assert np.isfinite(fix).all() and np.isfinite(risk).all()
    assert np.all((fix>=0)&(fix<=1)) and np.all((risk>=0)&(risk<=1))
    return fix,risk


def evaluate_prediction_arrays(arrays,classes):
    methods=arrays['methods'].tolist()
    conditions=arrays['conditions'].tolist()
    y=arrays['y']
    predictions,actions=arrays['predictions'],arrays['choices']
    assert methods==METHODS and conditions==[c[0] for c in CONDITIONS]
    assert predictions.shape==actions.shape==(len(methods),len(conditions),len(y))
    assert np.issubdtype(predictions.dtype,np.integer) and predictions.min()>=0 and predictions.max()<classes
    assert np.issubdtype(actions.dtype,np.integer) and actions.min()>=0
    original_correct=predictions[methods.index('keep')]==y[None,:]
    rows,summary,participants=[],[],[]
    for m,method in enumerate(methods):
        correct=predictions[m]==y[None,:]
        method_rows=[]
        for d,condition in enumerate(conditions):
            helpful=float((~original_correct[d]&correct[d]).mean())
            harmful=float((original_correct[d]&~correct[d]).mean())
            accuracy=float(correct[d].mean())
            assert abs(accuracy-original_correct[d].mean()-helpful+harmful)<1e-10
            method_rows.append({'method':method,'condition':condition,'windows':len(y),
                'macro_f1':macro_f1(y,predictions[m,d],classes),'accuracy':accuracy,
                'intervention_rate':float((actions[m,d]!=0).mean()),'helpful_rate':helpful,'harmful_rate':harmful})
        rows.extend(method_rows)
        for scope,names in SCOPES.items():
            part=[row for row in method_rows if row['condition'] in names]
            summary.append({'method':method,'scope':scope,'unique_test_windows':len(y),
                **{key:float(np.mean([row[key] for row in part])) for key in
                  ['macro_f1','accuracy','intervention_rate','helpful_rate','harmful_rate']}})
        if method=='oracle_diagnostic':
            continue
        for subject in np.unique(arrays['subjects']):
            use=arrays['subjects']==subject
            present=np.unique(y[use])
            for scope,names in SCOPES.items():
                f1s,accs,helps,harms,interventions=[],[],[],[],[]
                for condition in names:
                    d=conditions.index(condition)
                    cm=np.bincount(y[use]*classes+predictions[m,d,use],minlength=classes*classes).reshape(classes,classes)
                    denom=cm.sum(0)+cm.sum(1)
                    values=np.divide(2.*cm.diagonal(),denom,out=np.zeros(classes),where=denom>0)
                    f1s.append(float(values[present].mean()));accs.append(float(correct[d,use].mean()))
                    helps.append(float((~original_correct[d,use]&correct[d,use]).mean()))
                    harms.append(float((original_correct[d,use]&~correct[d,use]).mean()))
                    interventions.append(float((actions[m,d,use]!=0).mean()))
                participants.append({'subject':int(subject),'method':method,'scope':scope,'windows':int(use.sum()),
                    'classes_present':len(present),'macro_f1_present_classes':float(np.mean(f1s)),
                    'accuracy':float(np.mean(accs)),'helpful_rate':float(np.mean(helps)),
                    'harmful_rate':float(np.mean(harms)),'intervention_rate':float(np.mean(interventions))})
    return rows,summary,participants


def finish_run(job,run,arrays,selected,imported=False):
    dataset=job['dataset']
    signature=job_signature(job)
    rows,summary,participants=evaluate_prediction_arrays(arrays,len(protocol['datasets'][dataset]['class_names']))
    frames={name:pd.DataFrame([{**job,**row} for row in values]) for name,values in
            [('per_condition',rows),('test_summary',summary),('per_participant',participants)]}
    if imported:
        # Recompute every source summary metric from saved predictions before reusing the pilot.
        for filename,key,columns in [('notebook04_test_summary.csv','test_summary',['method','scope']),
                                     ('notebook04_per_participant.csv','per_participant',['subject','method','scope'])]:
            original=pd.read_csv(source_file('nb04',filename))
            original=original[original.dataset==dataset].set_index(columns).sort_index()
            computed=frames[key].set_index(columns).sort_index()
            assert original.index.equals(computed.index)
            for col in ['accuracy','helpful_rate','harmful_rate','intervention_rate']+(
                ['macro_f1'] if key=='test_summary' else ['macro_f1_present_classes']):
                assert np.allclose(original[col],computed[col],atol=1e-12,rtol=0),f'Pilot import parity failed: {col}'
    for name,frame in frames.items():
        save_csv(run/(name+'.csv'),frame)
    lookup={(r['method'],r['scope']):r for r in summary}
    comparisons=[]
    for scope in SCOPES:
        learner,baseline,keep=lookup['harm_aware_policy',scope],lookup[PRIMARY[dataset],scope],lookup['keep',scope]
        comparisons.append({**job,'scope':scope,'primary_comparator':PRIMARY[dataset],
            'keep_macro_f1':keep['macro_f1'],'comparator_macro_f1':baseline['macro_f1'],
            'policy_macro_f1':learner['macro_f1'],'delta_macro_f1_pp':100*(learner['macro_f1']-baseline['macro_f1']),
            'policy_harmful_rate':learner['harmful_rate'],'comparator_harmful_rate':baseline['harmful_rate'],
            'policy_intervention_rate':learner['intervention_rate'],
            'policy_helpful_rate':learner['helpful_rate'],
            'empirical_harm_le_1pct':bool(learner['harmful_rate']<=.01+1e-12)})
    save_csv(run/'primary_comparison.csv',pd.DataFrame(comparisons))
    main=selected['selected_methods']['harm_aware_policy']['settings']
    result={**job,'signature':signature,'study_signature':STUDY_SIGNATURE,'imported_pilot':imported,
        'unique_test_windows':len(arrays['y']),'test_participants':[int(x) for x in np.unique(arrays['subjects'])],
        'selected_main_policy':main,'structural_keep_only':bool(main['threshold']>=1 and main['harm_penalty']>=0),
        'primary_comparisons':comparisons,'test_threshold_adjustments':0,
        'clean_f1_loss_pp':100*(lookup['keep','clean']['macro_f1']-lookup['harm_aware_policy','clean']['macro_f1']),
        'full_candidate_generation_required':True}
    # Completion is committed only after all files used by the aggregate report are written and hashed.
    result['artifacts']=[{'filename':p.name,'sha256':file_sha256(p),'bytes':p.stat().st_size}
        for p in [run/name for name in ['test_predictions.npz','selected_policy.json','per_condition.csv',
                                      'test_summary.csv','per_participant.csv','primary_comparison.csv']]]
    save_json(run/'dataset_complete.json',result)
    print(f'  COMPLETED {identity(job)} | keep-only={result["structural_keep_only"]}',flush=True)
    return result


def import_pilot(job,run):
    source_run=identity(job)
    original=source_file('nb04',source_run+'/test_predictions.npz')
    with np.load(original,allow_pickle=False) as saved:
        assert saved['signature'].item()==prior_report['signature']
        arrays={key:saved[key] for key in ['methods','conditions','predictions','choices','y','subjects','window_ids','source_indices']}
    data=load_dataset(job,need_raw=False)
    indices=data['splits']['test']
    assert np.array_equal(arrays['source_indices'],indices)
    for key in ['y','subjects','window_ids']:
        assert np.array_equal(arrays[key],data[key][indices])
    selected=source_json('nb03',source_run+'/selected_policy.json')
    assert selected['selected_methods']['harm_aware_policy']['settings']==prior_by_dataset[job['dataset']]['selected_policy']
    save_json(run/'selected_policy.json',selected)
    save_npz(run/'test_predictions.npz',signature=np.asarray(job_signature(job)),**arrays)
    result=finish_run(job,run,arrays,selected,imported=True)
    print('  Original Notebook 04 summary and participant metrics reproduced exactly.',flush=True)
    return result


def train_and_evaluate(job,run):
    global SEED
    SEED=job['seed']  # Original head training uses run seed, plus one for the harm head.
    signature=job_signature(job)
    save_json(run/'run_configuration.json',{'signature':signature,'job':job,'study_signature':STUDY_SIGNATURE})
    data=load_dataset(job)
    groups,classes=data['groups'],data['classes']
    train_values,validation_values=normalized(data,'train'),normalized(data,'validation')
    stage_seed=SEED+(0 if job['dataset']=='dsads' else 100000)
    check_budget()
    seed_everything(stage_seed)
    classifier=ActivityCNN(train_values.shape[1],classes)
    classifier,_=fit_model('classifier',classifier,train_values,data['y'][data['splits']['train']],
        validation_values,data['y'][data['splits']['validation']],groups,classes,run,signature,stage_seed)
    check_budget()
    seed_everything(stage_seed+1000)
    reconstructor=SensorReconstructor(train_values.shape[1])
    reconstructor,_=fit_model('reconstructor',reconstructor,train_values,data['y'][data['splits']['train']],
        validation_values,data['y'][data['splits']['validation']],groups,classes,run,signature,stage_seed+1000)
    del train_values
    cache_signature=json_signature({'run':signature,'classifier':file_sha256(run/'classifier_best.pt'),
                                    'reconstructor':file_sha256(run/'reconstructor_best.pt')})
    policy=case_caches(job,'policy',normalized(data,'policy'),data,classifier,reconstructor,run,cache_signature)
    validation=case_caches(job,'validation',validation_values,data,classifier,reconstructor,run,cache_signature)
    del validation_values
    scale=np.maximum(np.quantile(policy['residuals'][0],.95,axis=0),1e-8)
    X,feature_names=policy_features(policy['probabilities'],policy['residuals'],policy['statistics'],scale)
    fix_target,harm_target=outcome_targets(policy)
    check_budget()
    benefit,_=fit_head('benefit',X,fix_target,run,signature,feature_names)
    check_budget()
    harm,_=fit_head('harm',X,harm_target,run,signature,feature_names)
    del X,fix_target,harm_target,policy
    fix,risk=head_outputs(validation,scale,feature_names,benefit,harm)
    val_choices=baseline_choices(validation,scale)
    clean_keep=aggregate_rows(evaluate_choices(validation,val_choices['keep'],'keep'))['clean_macro_f1']
    selected_methods,trials={},[]
    for family in FAMILIES:
        best,candidates=select_settings(validation,fix,risk,scale,family,clean_keep)
        selected_methods[family]={'settings':best['settings'],'validation_summary':best['summary'],
            'keep_only_on_validation':bool(np.all(best['actions']==0))}
        val_choices[family]=best['actions']
        trials.extend(candidates)
    selected={'signature':signature,'residual_scale':scale.tolist(),'feature_names':feature_names,
              'action_names':validation['action_names'].tolist(),'selected_methods':selected_methods,
              'selection_split':'validation','test_information_used_for_selection':False}
    selected_path=run/'selected_policy.json'
    if selected_path.exists():
        assert json.loads(selected_path.read_text())==selected,'Saved validation-selected policy changed.'
    else:
        save_json(selected_path,selected)
    save_csv(run/'selection_trials.csv',pd.DataFrame(trials))
    save_csv(run/'validation_comparison.csv',pd.DataFrame([row for method,actions in val_choices.items()
        for row in evaluate_choices(validation,actions,method)]))
    print('  Frozen validation-selected policy:',selected_methods['harm_aware_policy']['settings'],flush=True)
    # Test tensors are constructed only after the selection record has been committed.
    del validation,fix,risk,val_choices
    check_budget()
    test=case_caches(job,'test',normalized(data,'test'),data,classifier,reconstructor,run,cache_signature)
    del data,classifier,reconstructor
    if DEVICE.type=='cuda':
        torch.cuda.empty_cache()
    fix,risk=head_outputs(test,scale,feature_names,benefit,harm)
    choices=baseline_choices(test,scale)
    for family,item in selected_methods.items():
        choices[family]=choose_actions(test,fix,risk,scale,item['settings'])
    assert list(choices)==METHODS
    all_predictions=test['probabilities'].argmax(-1)
    arrays={key:test[key] for key in ['y','subjects','window_ids','source_indices']}
    arrays.update(methods=np.asarray(METHODS),conditions=np.asarray(test['conditions']),
        predictions=np.stack([np.take_along_axis(all_predictions,choices[m][...,None],axis=2).squeeze(2) for m in METHODS]),
        choices=np.stack([choices[m] for m in METHODS]))
    save_npz(run/'test_predictions.npz',signature=np.asarray(signature),**arrays)
    return finish_run(job,run,arrays,selected)


def completed_result(job):
    run=OUTPUT/'runs'/identity(job)
    path=run/'dataset_complete.json'
    if not path.is_file():
        return None
    result=json.loads(path.read_text())
    assert result['signature']==job_signature(job) and result['study_signature']==STUDY_SIGNATURE
    for record in result['artifacts']:
        assert file_sha256(run/record['filename'])==record['sha256'],'A completed result artifact changed.'
    return result


def publish_progress(final=False):
    results=[r for job in JOBS if (r:=completed_result(job)) is not None]
    if not results:
        print('No complete configurations yet. Completed training epochs are saved.',flush=True)
        return
    for name in ['per_condition','test_summary','per_participant','primary_comparison']:
        frame=pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/(name+'.csv')) for r in results],ignore_index=True)
        save_csv(OUTPUT/('notebook05_'+name+'.csv'),frame)
    summary=pd.read_csv(OUTPUT/'notebook05_test_summary.csv')
    numeric=['macro_f1','accuracy','intervention_rate','helpful_rate','harmful_rate']
    seed_rows=[]
    for keys,part in summary.groupby(['dataset','seed','method','scope'],sort=False):
        dataset,seed,method,scope=keys
        expected=4 if dataset=='dsads' else 1
        assert len(part)==part.fold.nunique()<=expected
        seed_rows.append({'dataset':dataset,'seed':int(seed),'method':method,'scope':scope,
            'folds_completed':len(part),'folds_required':expected,'complete_for_seed':len(part)==expected,
            **{col:float(part[col].mean()) for col in numeric}})
    seed_frame=pd.DataFrame(seed_rows)
    save_csv(OUTPUT/'notebook05_per_seed_summary.csv',seed_frame)
    aggregate=[]
    for (dataset,method,scope),part in seed_frame[seed_frame.complete_for_seed].groupby(['dataset','method','scope'],sort=False):
        row={'dataset':dataset,'method':method,'scope':scope,'complete_seeds':len(part),'required_seeds':len(SEEDS),
             'full_protocol_complete_for_dataset':len(part)==len(SEEDS)}
        for col in numeric:
            row[col+'_mean']=float(part[col].mean())
            row[col+'_seed_sd']=float(part[col].std(ddof=1)) if len(part)>1 else None
        aggregate.append(row)
    aggregated=pd.DataFrame(aggregate)
    save_csv(OUTPUT/'notebook05_aggregate_summary.csv',aggregated)
    ready=len(results)==len(JOBS)
    report={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,'study_complete':ready,
            'configurations_complete':len(results),'configurations_required':len(JOBS),
            'imported_prior_configurations':sum(r['imported_pilot'] for r in results),
            'pending_configurations':[job for job in JOBS if identity(job) not in {identity(r) for r in results}],
            'completed_results':results,'aggregation':plan['aggregation'],
            'seed_sd_is_not_ci':True,'seeds_and_corrupted_views_are_not_independent_people':True,
            'no_significance_claim':True,'test_based_threshold_changes':0,
            'primary_comparators':PRIMARY,'output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook05_run_report.json',report)
    # The full checkpoint tree is intentionally not rehashed every time a partial summary is refreshed.
    inventory=[{'filename':str(p.relative_to(OUTPUT)),'sha256':file_sha256(p),'bytes':p.stat().st_size}
        for p in sorted(OUTPUT.glob('notebook05_*')) if p.is_file() and not p.name.endswith('.partial')
        and p.name!='notebook05_artifact_inventory.json']
    save_json(OUTPUT/'notebook05_artifact_inventory.json',inventory)
    if not final:
        print(f'Progress saved: {len(results)}/{len(JOBS)} configurations complete.',flush=True)
        return
    print('\nNOTEBOOK 05 '+('COMPLETE' if ready else 'PAUSED — RERUN THE SAME CELL'),flush=True)
    print(f'Configurations complete: {len(results)}/{len(JOBS)} | imported: {report["imported_prior_configurations"]}',flush=True)
    if not aggregated.empty:
        show=aggregated[aggregated.method.isin(['keep','residual_gate_mask','confidence_select','harm_aware_policy'])].copy()
        show['macro_f1_mean']*=100.;show['macro_f1_seed_sd']*=100.;show['harmful_rate_mean']*=100.
        print('AVAILABLE COMPLETE-SEED SUMMARIES | percentages; SD across seeds is descriptive',flush=True)
        print(show[['dataset','method','scope','complete_seeds','macro_f1_mean','macro_f1_seed_sd','harmful_rate_mean']]
              .to_string(index=False,float_format=lambda x:f'{x:.2f}'),flush=True)
    else:
        print('Whole-seed DSADS coverage is still incomplete. Per-run results are saved.',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('Share this final output and notebook05_aggregate_summary.csv, notebook05_run_report.json, notebook05_primary_comparison.csv.',flush=True)


def main():
    # Import both existing runs first, even when the first DSADS seed is not yet complete.
    ordered=[job for job in JOBS if job['fold']=='fold_0' and job['seed']==2026]
    ordered += [job for job in JOBS if job not in ordered]
    try:
        for job in ordered:
            if completed_result(job) is not None:
                print('REUSE',identity(job),flush=True)
                continue
            check_budget()
            run=OUTPUT/'runs'/identity(job)
            (run/'action_cache').mkdir(parents=True,exist_ok=True)
            print('\nRUN',identity(job),flush=True)
            if job['fold']=='fold_0' and job['seed']==2026:
                import_pilot(job,run)
            else:
                train_and_evaluate(job,run)
            gc.collect()
            if DEVICE.type=='cuda':
                torch.cuda.empty_cache()
            publish_progress()
    except PauseStudy as exc:
        print('\n',str(exc),flush=True)
    finally:
        publish_progress(final=True)


with threadpool_limits(limits=2):
    main()


Mounted at /content/drive
NOTEBOOK 05 | original full protocol | one executable cell
Device: Tesla T4 | PyTorch: 2.11.0+cu130
DSADS: 4 folds x 3 seeds | HAR: official test x 3 seeds | 15 total configurations.
Completed fold_0 / seed 2026 results will be imported. Training and selection rules remain fixed.
Full study plan frozen: c49ce0a17a8cd793

RUN dsads_fold_0_seed_2026
  COMPLETED dsads_fold_0_seed_2026 | keep-only=False
  Original Notebook 04 summary and participant metrics reproduced exactly.
Progress saved: 1/15 configurations complete.

RUN har_fold_0_seed_2026
  COMPLETED har_fold_0_seed_2026 | keep-only=True
  Original Notebook 04 summary and participant metrics reproduced exactly.
Progress saved: 2/15 configurations complete.

RUN dsads_fold_1_seed_2026
  classifier epoch 01/30 | loss 1.3674 | validation mean Macro-F1 0.7324
  classifier epoch 05/30 | loss 0.1516 | validation mean Macro-F1 0.8061
  classifier epoch 10/30 | loss 0.1140 | validation mean Macro-F1 0.8502
  clas